**Author:** Javier Lendínez  
**License:** GNU Affero General Public License v3.0 ([LICENSE](../../LICENSE))

# Persistence and Performance Verification

This notebook follows the procedure in `docs/README-performance.md` to check result reuse and the persistence formats implemented in `src/vlm/lib/vlm.py`. I compare cache indicators, verify dimensions and finite values, and perform save/load round trips in a temporary directory.

This is software verification, not additional aerodynamic validation beyond the earlier cases: a fast run or a successful file read does not, by itself, demonstrate the physical accuracy of the method.

In [74]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    current = start
    while current != current.parent:
        if (current / 'src' / 'vlm').is_dir():
            return current
        current = current.parent
    raise RuntimeError('Could not locate the VLMPy repository root (expected src/vlm).')


REPO_ROOT = _find_repo_root(Path.cwd())
SRC_VLM = REPO_ROOT / 'src' / 'vlm'
if str(SRC_VLM) not in sys.path:
    sys.path.insert(0, str(SRC_VLM))

print(f'VLM source path added to sys.path: {SRC_VLM}')

VLM source path added to sys.path: c:\Users\javil\OneDrive - Universidad Alfonso X el Sabio\Documents\GitHub\VLMPy\src\vlm


In [75]:
import contextlib
import io
import tempfile
from pathlib import Path

import numpy as np

from lib.vlm import VLM


def _quiet_run(v):
    """Run vlm.run_vlm() while discarding the library's internal diagnostic prints."""
    out = io.StringIO()
    with contextlib.redirect_stdout(out):
        v.run_vlm()

## 1. Reference Case

I define a small trapezoidal wing so the resolution and checks can be repeated quickly. Its parameters remain fixed throughout the cache tests.

In [86]:
plane = {
    'wing_sections': [
        {
            'chord_root': 1.0,
            'chord_tip': 0.7,
            'span_fraction': 6.0,
            'sweep': np.radians(10.0),
            'dihedral': np.radians(3.0),
            'NACA_root': '2412',
            'NACA_tip': '2412',
        }
    ]
}

vlm = VLM(plane=plane, u=40.0, rho=1.225, alpha=np.radians(3.0), beta=0.0, n=16, m=6)

## 2. Cache Reuse and Invalidation

`run_vlm()` runs the calculation chain once: it checks geometry, then discretization, and finally solves the flow. For a new instance, I expect cache misses (`False`) on the first run and cache hits (`True`) when the exact same inputs are repeated.

Changing `alpha` changes the geometry key because the wing geometry is rotated by that angle. The first run with the new value should invalidate geometry, discretization, the influence matrix, and the solution; the next unchanged run should reuse them. These flags verify reuse, not a measured speedup.

In [87]:
_quiet_run(vlm)
first_timings = dict(vlm.timings)
print('First run, new inputs:')
for key, value in first_timings.items():
    if key.endswith('_hit'):
        print(f'  {key}: {value}')

assert first_timings['geometry_cache_hit'] is False
assert first_timings['discretization_cache_hit'] is False
assert first_timings['influence_cache_hit'] is False
assert first_timings['solution_cache_hit'] is False

_quiet_run(vlm)
second_timings = dict(vlm.timings)
print('Second run, unchanged inputs:')
for key, value in second_timings.items():
    if key.endswith('_hit'):
        print(f'  {key}: {value}')

assert second_timings['geometry_cache_hit'] is True
assert second_timings['discretization_cache_hit'] is True
assert second_timings['influence_cache_hit'] is True
assert second_timings['solution_cache_hit'] is True

First run, new inputs:
  geometry_cache_hit: False
  discretization_cache_hit: False
  influence_cache_hit: False
  solution_cache_hit: False
Second run, unchanged inputs:
  geometry_cache_hit: True
  discretization_cache_hit: True
  influence_cache_hit: True
  solution_cache_hit: True


In [88]:
# Changing alpha should invalidate geometry, discretization, influence, and solution.
vlm.alpha = np.radians(6.0)
_quiet_run(vlm)
after_change = dict(vlm.timings)
print('First run after changing alpha:')
for key, value in after_change.items():
    if key.endswith('_hit'):
        print(f'  {key}: {value}')

assert after_change['geometry_cache_hit'] is False
assert after_change['discretization_cache_hit'] is False
assert after_change['influence_cache_hit'] is False
assert after_change['solution_cache_hit'] is False

_quiet_run(vlm)
after_repeat = dict(vlm.timings)
assert after_repeat['geometry_cache_hit'] is True
assert after_repeat['discretization_cache_hit'] is True
assert after_repeat['influence_cache_hit'] is True
assert after_repeat['solution_cache_hit'] is True
print('Changing alpha invalidates the caches; the next repeat reuses them.')

First run after changing alpha:
  geometry_cache_hit: False
  discretization_cache_hit: False
  influence_cache_hit: False
  solution_cache_hit: False
Changing alpha invalidates the caches; the next repeat reuses them.


## 3. Dimension and Finiteness Checks

I repeat the checks from `docs/README-performance.md`: the influence matrix must have one row and one column per panel, and the main numerical quantities must be finite.

In [79]:
assert np.isfinite(vlm.P_ij).all()
assert np.isfinite(vlm.gammas).all()
assert np.isfinite(vlm.CL)
assert np.isfinite(vlm.CD)
assert vlm.P_ij.shape == (len(vlm.panel_data), len(vlm.panel_data))
print('All finite-value and shape checks passed (docs/README-performance.md procedure).')

All finite-value and shape checks passed (docs/README-performance.md procedure).


## 4. Configuration Export and Checksum

`export_configuration` creates a serializable snapshot of the geometry and, optionally, the flight conditions. `configuration_checksum` calculates a SHA-256 checksum of canonical JSON; I use it to identify changes in the input data.

In [80]:
configuration = vlm.export_configuration(include_flight=True)
checksum = VLM.configuration_checksum(configuration)

print(f'Configuration checksum (SHA-256): {checksum}')
print(f'Flight parameters recorded: {configuration["flight_parameters"]}')

Configuration checksum (SHA-256): e9f0e8137580509e8e759a09d7ea8b2f853f01e8b4282b3047dfc405eb862261
Flight parameters recorded: {'u': 40.0, 'rho': 1.225, 'alpha': 0.10471975511965978, 'beta': 0.0, 'n': 16, 'm': 6}


## 5. `save_results` / `load_results` Round Trip

I save the results and their associated NumPy data in a temporary folder. This keeps the example from writing to `src/vlm/data/saved_states/` or altering the user's stored cases.

In [81]:
with tempfile.TemporaryDirectory() as tmp_dir:
    results_path = Path(tmp_dir) / 'demo_case.json'
    manifest = vlm.save_results(str(results_path))
    print(f'Manifest schema: {manifest["schema"]} v{manifest["schema_version"]}')
    print(f'Saved summary: {manifest["summary"]}')

    vlm_reloaded = VLM(plane=plane, u=40.0, rho=1.225, alpha=vlm.alpha, beta=0.0, n=16, m=6)
    _out = io.StringIO()
    with contextlib.redirect_stdout(_out):
        vlm_reloaded.calculate_geometry()
        vlm_reloaded.calculate_discretization()
    vlm_reloaded.load_results(str(results_path))

    print(f'Original CL={vlm.CL:.6f}  Reloaded CL={vlm_reloaded.CL:.6f}')
    assert np.isclose(vlm.CL, vlm_reloaded.CL)
    assert np.isclose(vlm.CD, vlm_reloaded.CD)
    print('save_results / load_results round trip verified.')

Manifest schema: vlmpy.results v1
Saved summary: {'CL': 0.6461622511022282, 'CD': 0.01629872706104148, 'lift': 3229.518931008937, 'drag': 81.46103785108534}
Original CL=0.646162  Reloaded CL=0.646162
save_results / load_results round trip verified.


## 6. `save_plane` / `load_plane` and `save_state` / `load_state` Round Trips

`save_plane` and `load_plane` exchange geometry as JSON. `save_state` and `load_state` serialize the object's internal dictionary with `pickle`; this format depends on the software version and environment, and must never be loaded from an untrusted source.

In [82]:
with tempfile.TemporaryDirectory() as tmp_dir:
    plane_path = Path(tmp_dir) / 'plane.json'
    vlm.save_plane(str(plane_path))

    vlm_plane_check = VLM(plane={}, u=40.0, rho=1.225, alpha=0.0, beta=0.0, n=16, m=6)
    vlm_plane_check.load_plane(str(plane_path))
    assert vlm_plane_check.plane['wing_sections'][0]['NACA_root'] == '2412'
    print('save_plane / load_plane round trip verified.')

    state_path = Path(tmp_dir) / 'state.pkl'
    vlm.save_state(str(state_path))

    vlm_state_check = VLM(plane={}, u=0.0, rho=0.0, alpha=0.0, beta=0.0, n=1, m=1)
    vlm_state_check.load_state(str(state_path))
    assert np.isclose(vlm_state_check.CL, vlm.CL)
    print('save_state / load_state round trip verified.')

save_plane / load_plane round trip verified.
save_state / load_state round trip verified.


## 7. Conclusions and Limitations

The first run misses the geometry, discretization, influence, and solution caches; an identical repeat reuses all four stages. Changing `alpha` invalidates all four, and a subsequent run at the same angle reuses them. These flags confirm internal behavior, not a measured speedup.

For the reference case, after changing to $\alpha=6^\circ$, I obtained $C_L=0.646162$ and $C_D=0.016299$. The JSON/NPZ round trip restored the same coefficients; the plane JSON and `pickle` state round trips also passed. These checks verify persistence, not the aerodynamic validity of the configuration.

- `save_state` / `load_state` uses `pickle`, is version-dependent, and must not load files from untrusted sources.
- The influence matrix and circulation are numerical diagnostics, not surface-pressure maps.
- Persistence preserves computed results but does not replace checking whether the physical configuration is appropriate.

### Visual Diagnosis of the Solution

To complement the shape and finiteness checks, I plot the magnitude of the influence matrix $P_{ij}$ and the solved circulation by panel index. The dense matrix shows coupling between control points and vortices; the second figure helps inspect circulation variation. Neither plot is a pressure map.

In [89]:
import contextlib
import io
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

current = Path.cwd()
while current != current.parent and not (current / 'src' / 'vlm').is_dir():
    current = current.parent
if not (current / 'src' / 'vlm').is_dir():
    raise RuntimeError('Could not locate the VLMPy repository root.')
sys.path.insert(0, str(current / 'src' / 'vlm'))

from lib.vlm import VLM

diagnostic_plane = {
    'wing_sections': [{
        'chord_root': 1.0, 'chord_tip': 0.7,
        'span_fraction': 6.0, 'sweep': np.radians(10.0),
        'dihedral': np.radians(3.0),
        'NACA_root': '2412', 'NACA_tip': '2412',
    }]
}
diagnostic_case = VLM(
    diagnostic_plane, 40.0, 1.225, np.radians(3.0), 0.0, 10, 4
)
captured = io.StringIO()
with contextlib.redirect_stdout(captured):
    diagnostic_case.run_vlm()

influence_figure = go.Figure()
influence_figure.add_trace(go.Heatmap(
    z=np.abs(diagnostic_case.P_ij),
    colorbar={'title': '|P_ij|'},
))
influence_figure.update_layout(
    title='Magnitude of the influence matrix',
    xaxis_title='Source panel j',
    yaxis_title='Control point i',
)
influence_figure.show()

gamma_values = np.ravel(diagnostic_case.gammas)
gamma_figure = go.Figure()
gamma_figure.add_trace(go.Scatter(
    x=np.arange(len(gamma_values)), y=gamma_values,
    mode='lines+markers', name='Circulation',
))
gamma_figure.update_layout(
    title='Solved circulation by panel',
    xaxis_title='Panel index',
    yaxis_title='Gamma [m^2/s]',
    template='plotly_white',
)
gamma_figure.show()

assert influence_figure.data[0].z.shape == diagnostic_case.P_ij.shape
assert np.isfinite(gamma_values).all()

## 8. Runtime by Stage and Mesh Size

I measure wall-clock time for one cold run and the stage timings reported by `vlm.timings` at several mesh sizes. I create a new instance for each case to avoid mixing caches across resolutions. The figure helps locate the dominant cost for this machine and configuration; one sample per mesh cannot establish a general performance trend or support comparison with another implementation.

In [90]:
import contextlib
import io
import sys
import time
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

current = Path.cwd()
while current != current.parent and not (current / 'src' / 'vlm').is_dir():
    current = current.parent
if not (current / 'src' / 'vlm').is_dir():
    raise RuntimeError('Could not locate the VLMPy repository root.')
sys.path.insert(0, str(current / 'src' / 'vlm'))

from lib.vlm import VLM

timing_plane = {
    'wing_sections': [{
        'chord_root': 1.0, 'chord_tip': 0.7, 'span_fraction': 6.0,
        'sweep': np.radians(10.0), 'dihedral': np.radians(3.0),
        'NACA_root': '2412', 'NACA_tip': '2412',
    }]
}
requested_n_values = [8, 12, 16, 20]
m_timing = 4
stages = ['geometry', 'discretization', 'influence_matrix', 'linear_solve', 'postprocess']
timing_by_stage = {stage: [] for stage in stages}
wall_times = []
panel_counts = []

for requested_n in requested_n_values:
    case = VLM(timing_plane, 40.0, 1.225, np.radians(3.0), 0.0, requested_n, m_timing)
    captured = io.StringIO()
    start_time = time.perf_counter()
    with contextlib.redirect_stdout(captured):
        case.run_vlm()
    wall_time = time.perf_counter() - start_time
    wall_times.append(wall_time)
    panel_counts.append(len(case.panel_data))
    for stage in stages:
        timing_by_stage[stage].append(float(case.timings[stage]))
    print(
        'n requested={}, n effective={}, m effective={}, panels={}, wall time={:.6f} s'.format(
            requested_n, case.n, case.m, len(case.panel_data), wall_time
        )
    )

timing_figure = go.Figure()
for stage in stages:
    timing_figure.add_trace(go.Bar(
        x=panel_counts, y=timing_by_stage[stage], name=stage,
    ))
timing_figure.update_layout(
    title='Stage timings for one cold run',
    xaxis_title='Effective panel count',
    yaxis_title='Reported time [s]',
    barmode='group', template='plotly_white',
)
timing_figure.show()

wall_figure = go.Figure()
wall_figure.add_trace(go.Scatter(
    x=panel_counts, y=wall_times, mode='lines+markers', name='Wall-clock time',
))
wall_figure.update_layout(
    title='Wall-clock time by mesh size',
    xaxis_title='Effective panel count',
    yaxis_title='Wall-clock time [s]', template='plotly_white',
)
wall_figure.show()

assert np.isfinite(wall_times).all()
for stage in stages:
    assert np.isfinite(timing_by_stage[stage]).all()

n requested=8, n effective=8, m effective=4, panels=32, wall time=0.005015 s
n requested=12, n effective=12, m effective=4, panels=48, wall time=0.007360 s
n requested=16, n effective=16, m effective=4, panels=64, wall time=0.009646 s
n requested=20, n effective=20, m effective=4, panels=80, wall time=0.010487 s


### Interpretation of the Measured Profile

In this run, with $m=4$, the effective mesh sizes of 32, 48, 64, and 80 panels gave wall-clock times of 5.36, 7.50, 11.63, and 15.37 ms. Among the reported stage timings, discretization was the most expensive (4.02 to 11.00 ms), followed by influence-matrix assembly (1.22 to 4.20 ms).

The observed trend suggests prioritizing those stages when profiling this case, but it does not establish a scaling law or an improvement over another implementation. Each mesh size was measured once in a single environment; a performance comparison would require repeated runs, medians, and documented execution conditions.